# Clasificaci?n con Dataset IRIS

In [ ]:
import pandas as pd 
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.datasets import load_iris

# 1) Cargar datos en X,y - DATASET IRIS
iris = load_iris()
# Crear DataFrame con nombres apropiados
df_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
df_iris['species'] = iris.target
df_iris['species_name'] = [iris.target_names[i] for i in iris.target]  # Para mostrar los nombres

display(df_iris.head().style.set_caption("<h3>Primeras filas del dataset Iris<h3>"))

target = "species"  
X = df_iris.drop(columns=[target, 'species_name'])  # Eliminamos species y species_name
y = df_iris[target]  # Target: especies (0, 1, 2)

display(pd.DataFrame(X.head()).style.set_caption("<h3>Caracter?sticas (X)<h3>"))

display(pd.DataFrame({
    target: iris.target_names,
    'C?digo': range(len(iris.target_names))
}).style.set_caption("<h3>Target (y) <h3>"))
        
display(pd.DataFrame({
    'Variables': X.columns.tolist()
}).style.set_caption("<h3>Nombre de las caracter?sticas <h3>").hide(axis='index'))

In [ ]:
# 2) Codificar y normalizar
# Para Iris no necesita codificaci?n porque ya son num?ricas
# Las caracter?sticas ya est?n en escala similar (cm)

# 3) Separar
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 4) Seleccion de modelo(s) 
bosque = RandomForestClassifier(random_state=42)

# 5) Entrenar
bosque.fit(X_train, y_train)

# 6) Probar y evaluar
y_pred = bosque.predict(X_test)

# Resultados en TEST
# Matriz de confusi?n
display(pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    columns=iris.target_names,
    index=iris.target_names
).style.set_caption("<h4>Matriz de confusi?n</h4>"))

# Reporte de clasificaci?n
display(pd.DataFrame(
    classification_report(y_test, y_pred, target_names=iris.target_names, output_dict=True)
).transpose().style.set_caption("<h4> Reporte de clasificaci?n </h4>"))

In [ ]:
# 7) Validaci?n Cruzada
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "accuracy": "accuracy",
    "f1_macro": "f1_macro",
    "roc_auc": "roc_auc_ovr",
    "average_precision": "average_precision",
}

scores = cross_validate(bosque, X_train, y_train, cv=cv, scoring=scoring, return_train_score=True, n_jobs=-1)

In [ ]:
# Todas las m?tricas en una sola tabla
display(pd.DataFrame({
    'M?trica': [
        'Exactitud Validaci?n',
        'Exactitud Entrenamiento', 
        'F1-Score Macro',
        'ROC AUC Medio',
        'Average Precision Medio'
    ],
    'Valor': [
        f"{scores['test_accuracy'].mean():.3f} ? {scores['test_accuracy'].std():.3f}",
        f"{scores['train_accuracy'].mean():.3f} ? {scores['train_accuracy'].std():.3f}",
        f"{scores['test_f1_macro'].mean():.3f} ? {scores['test_f1_macro'].std():.3f}",
        f"{scores['test_roc_auc'].mean():.3f}",
        f"{scores['test_average_precision'].mean():.3f}"
    ]
}).style.set_caption("<h3>VALIDACI?N CRUZADA (5-FOLD)</h3>").hide(axis='index'))

In [ ]:
# Todas las m?tricas en una sola tabla
display(pd.DataFrame({
    'M?trica': [
        'M?tricas Principales',
        'Exactitud Validaci?n',
        'Exactitud Entrenamiento', 
        'F1-Score Macro',
        'ROC AUC Medio',
        'Average Precision Medio'
    ],
    'Valor': [
        f"ACC {scores['test_accuracy'].mean():.3f} ? {scores['test_accuracy'].std():.3f} | F1 {scores['test_f1_macro'].mean():.3f} ? {scores['test_f1_macro'].std():.3f}",
        f"{scores['test_accuracy'].mean():.3f} ? {scores['test_accuracy'].std():.3f}",
        f"{scores['train_accuracy'].mean():.3f} ? {scores['train_accuracy'].std():.3f}",
        f"{scores['test_f1_macro'].mean():.3f} ? {scores['test_f1_macro'].std():.3f}",
        f"{scores['test_roc_auc'].mean():.3f}",
        f"{scores['test_average_precision'].mean():.3f}"
    ]
}).style.set_caption("<h3>VALIDACI?N CRUZADA (5-FOLD)</h3>").hide(axis='index'))

In [ ]:
# 6) Probar (con el conjunto de test)
y_pred = bosque.predict(X_test)

# 7) Evaluar en el conjunto de entrenamiento
# Reporte y Matriz de Confusi?n
display(pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    columns=iris.target_names,
    index=iris.target_names
).style.set_caption("<h4>Matriz de Confusi?n</h4>"))

display(pd.DataFrame(
    classification_report(y_test, y_pred, target_names=iris.target_names, output_dict=True)
).transpose().style.set_caption("<h4>Reporte de Clasificaci?n</h4>"))

display(pd.DataFrame({
    'M?trica': ['Score del Modelo'],
    'Valor': [f"{bosque.score(X_test, y_test):.3f}"]
}).style.set_caption("<h4>Evaluaci?n Final</h4>").hide(axis='index'))

# Resumen

In [ ]:
display(pd.DataFrame({
    'M?trica': ['Accuracy Global', 'Validaci?n Cruzada (ACC)', 'F1-Score Macro', 'ROC AUC'],
    'Valor': ['88.9%', '95.2% ? 4.3%', '95.2% ? 4.3%', '99.8%'],
    'Evaluaci?n': [
        "Alta efectividad en clasificaci?n",
        "Excelente consistencia entre folds", 
        "Desempe?o balanceado entre clases",
        "Capacidad discriminativa excepcional"
    ]
}).style.set_caption("<h1>?? RESUMEN EJECUTIVO DE PERFORMANCE<h1>").hide(axis='index'))

display(pd.DataFrame({
    'Especie': ['setosa', 'versicolor', 'virginica'],
    'Precisi?n': ['100%', '77.8%', '91.7%'],
    'Recall': ['100%', '93.3%', '73.3%'], 
    'F1-Score': ['100%', '84.8%', '81.5%'],
    'Diagn?stico': [
        "Clasificaci?n perfecta",
        "Buena detecci?n con algunas imprecisiones",
        "Alta precisi?n con algunos no detectados"
    ]
}).style.set_caption("<h3>?? AN?LISIS DETALLADO POR ESPECIE<h3>").hide(axis='index'))

display(pd.DataFrame({
    'Especie': ['setosa', 'versicolor', 'virginica'],
    'Comportamiento': [
        "15/15 correctos - Clasificaci?n perfecta",
        "14/15 correctos - 1 confundido con virginica", 
        "11/15 correctos - 4 confundidos con versicolor"
    ],
    'Patr?n de Error': [
        "Ning?n error de clasificaci?n",
        "Leve confusi?n con especie virginica",
        "Confusi?n moderada con especie versicolor"
    ]
}).style.set_caption("<h3>?? INTERPRETACI?N MATRIZ DE CONFUSI?N<h3>").hide(axis='index'))

display(pd.DataFrame({
    'Aspecto': ['Generalizaci?n', 'Sobreajuste', 'Consistencia', 'Capacidad Discriminativa'],
    'Resultado': [
        "95.2% de efectividad en datos no vistos",
        "Brecha m?nima entrenamiento vs validaci?n", 
        "Alta consistencia (?4.3% entre folds)",
        "Excepcional (ROC AUC 99.8%)"
    ],
    'Evaluaci?n': [
        "Excelente capacidad de generalizaci?n",
        "Modelo bien ajustado sin sobreajuste",
        "Performance estable y confiable",
        "Separa casi perfectamente las clases"
    ]
}).style.set_caption("<h3>?? AN?LISIS DE VALIDACI?N CRUZADA<h3>").hide(axis='index'))

display(pd.DataFrame({
    'Hallazgo': [
        "Clasificaci?n perfecta para setosa",
        "Confusi?n entre versicolor y virginica", 
        "Performance de validaci?n superior a test",
        "Capacidad discriminativa excepcional"
    ],
    'Implicaci?n': [
        "El modelo identifica sin errores la especie setosa",
        "Las especies versicolor y virginica presentan similitudes",
        "El modelo generaliza mejor de lo que sugiere el test ?nico",
        "El modelo casi perfecto separando las clases"
    ]
}).style.set_caption("<h3>?? CONCLUSIONES Y HALLAZGOS PRINCIPALES<h3>").hide(axis='index'))